# Week 4 · Day 1 — JavaScript essentials

*HTML is the structure, CSS is the look — JavaScript is the **behavior**. It's what makes a Claude artifact actually do something.*

**By the end you'll have shipped:** a small **JavaScript module** that computes a matters summary — variables, functions, and the `map`/`filter`/`reduce` trio that powers React — run for real with **Node.js**, plus a mini interactive page you open in a browser.

### 📋 Lesson card

| | |
|---|---|
| **Module** | M6 · Web UI (Week 4 — JavaScript & React) |
| **Prerequisites** | Week 3 (HTML/CSS); Week 1 Python helps (we compare the two) |
| **Est. time** | ~30 min |
| **Capstone slice** | The **behavior** layer of the *Matter Intelligence* UI |
| **Difficulty** | Core + `Go Deeper 🔧` |
| **Runs offline?** | ✅ Yes — JS runs locally via **Node.js** (no internet); degrades gracefully if Node is missing |

### 🎯 Learning objectives

By the end you'll be able to:
- Read and write JavaScript **variables** (`const`/`let`), **strings** (template literals), and **functions** (arrow `=>`).
- Work with **arrays** and **objects** — the shape of all web data.
- Use **`map`**, **`filter`**, and **`reduce`** — the three methods behind every React list.
- Recognize **DOM** access and **events** (`onclick`) — how a page reacts to a user.
- Map each JS idea to the **Python** you already know.

### ⚖️ Why it matters

Every interactive Claude artifact has a `<script>` — the JavaScript that filters a table, updates a total, toggles a theme. In Week 3 you learned to *recognize* it; now you learn to *read and edit* it. And one JS pattern matters above all: **`array.map(...)`** — turning a list of data into a list of UI. That single idea *is* how React renders your matters, so it's the bridge into tomorrow.

### ⚙️ Setup

`run_js(code)` runs a JavaScript snippet with **Node.js** and prints its output — real execution, offline. (JavaScript can't run inside the notebook's HTML output, so we use Node, the same engine behind the build tools you'll meet Thursday.) `show()` is back for the browser bits.

In [ ]:
import os, subprocess, shutil
from IPython.display import HTML, display

NODE = shutil.which("node")

def run_js(code: str):
    """Run a JavaScript snippet with Node.js and print its output."""
    if not NODE:
        print("⚠️  Node.js not found — install it from https://nodejs.org to run JS live.")
        print("   (Week 4 needs Node.) The code that would run:\n")
        print(code)
        return
    env = {k: v for k, v in os.environ.items() if k != "FORCE_COLOR"}   # Jupyter sets FORCE_COLOR; drop it
    env["NODE_DISABLE_COLORS"] = "1"; env["NO_COLOR"] = "1"             # keep Node's output plain text
    result = subprocess.run([NODE, "-e", code], capture_output=True, text=True, timeout=30, env=env)
    print(result.stdout, end="")
    if result.stderr:
        print(result.stderr, end="")

def show(html: str):
    display(HTML(html))

print(f"✅ Node.js {'found: ' + subprocess.run([NODE,'--version'],capture_output=True,text=True).stdout.strip() if NODE else 'NOT found (examples will print instead of run)'}")
run_js("console.log('Hello from JavaScript 👋')")

### 1 · JavaScript next to Python — a quick orientation

You already think in code. JavaScript differs from Python in surface syntax, not spirit:

| | Python | JavaScript |
|---|---|---|
| declare a variable | `x = 5` | `const x = 5;` (or `let` if it changes) |
| print | `print(x)` | `console.log(x);` |
| block | indentation + `:` | `{ … }` and `;` |
| comment | `# …` | `// …` |

> **Legal analogy:** same body of law, different jurisdiction's phrasing. The *concepts* transfer; the punctuation changes.

In [ ]:
run_js("""
const firm = "Rivera & Associates";   // const = can't be reassigned
let matterCount = 3;                   // let  = can change later
matterCount = matterCount + 1;
console.log(firm);
console.log("Open matters:", matterCount);
""")

**What just happened:** `const` declares a value that won't be reassigned; `let` one that will. Statements end with `;`, blocks use `{ }`. `console.log` is JavaScript's `print`.

### 2 · Strings & template literals

Regular strings use quotes. **Template literals** use backticks `` ` `` and let you drop variables in with `${…}` — JavaScript's f-string.

In [ ]:
run_js("""
const client = "Brightline LLC";
const amount = 42750.5;
// backticks + ${...}  ==  Python f-string
console.log(`Matter for ${client}: $${amount.toFixed(2)}`);
""")

**What just happened:** `` `…${client}…` `` interpolated the variable, and `amount.toFixed(2)` formatted the number to 2 decimals — the JS twin of Python's `f"{client}"` and `f"{amount:.2f}"`.

### 3 · Functions — the arrow form you'll see everywhere

JavaScript has two function styles. Claude's code overwhelmingly uses the **arrow** form `(args) => { … }`, so learn to read it.

In [ ]:
run_js("""
// classic form
function feeAfterDiscount(amount, pct) {
  return amount * (1 - pct);
}

// arrow form — same thing, the style you'll see in artifacts
const feeArrow = (amount, pct) => amount * (1 - pct);

console.log(feeAfterDiscount(1000, 0.1));   // 900
console.log(feeArrow(1000, 0.1));           // 900
""")

**What just happened:** both functions return a discounted fee. The arrow `(amount, pct) => amount * (1 - pct)` is a compact function; when the body is a single expression, it returns it automatically (no `return` needed). You'll see arrows constantly next lesson.

### 4 · Arrays & objects — the shape of web data

An **object** `{ key: value }` is a record (Python `dict`); an **array** `[ … ]` is an ordered list (Python `list`). Web data is almost always **an array of objects** — exactly like a list of matter records.

In [ ]:
run_js("""
const matters = [
  { id: "M-1002", client: "Brightline LLC", area: "Litigation",  billed: 42750.5, active: true  },
  { id: "M-1001", client: "Acme Corp",      area: "Contracts",   billed: 18500.0, active: true  },
  { id: "M-1003", client: "Cedar Holdings", area: "M&A",         billed: 131200.0, active: false },
];

console.log("count:", matters.length);
console.log("first client:", matters[0].client);   // dot access, 0-based
""")

**What just happened:** `matters` is an array of objects; `matters.length` is its size, `matters[0].client` reads a field of the first record. This `[{...}, {...}]` shape is what an API returns and what a React list renders.

### 5 · `map`, `filter`, `reduce` — the three you must know

These transform an array without a loop. They're the JS twins of pandas / list comprehensions — **and `map` is literally how React turns data into UI.**

- **`filter`** → keep some rows (like `df[df.active]`)
- **`map`** → transform each row (like a comprehension)
- **`reduce`** → collapse to one value (like `.sum()`)

In [ ]:
run_js("""
const matters = [
  { id: "M-1002", client: "Brightline LLC", billed: 42750.5, active: true  },
  { id: "M-1001", client: "Acme Corp",      billed: 18500.0, active: true  },
  { id: "M-1003", client: "Cedar Holdings", billed: 131200.0, active: false },
];

// filter: only active matters
const active = matters.filter(m => m.active);
console.log("active count:", active.length);

// map: turn each matter into a one-line label
const labels = matters.map(m => `${m.id} — ${m.client}`);
console.log(labels);

// reduce: total billed across all matters
const total = matters.reduce((sum, m) => sum + m.billed, 0);
console.log("total billed:", total.toFixed(2));
""")

**What just happened:** `filter` kept the active matters, `map` turned each record into a label string, `reduce` summed the `billed` field (starting from `0`). Hold onto `map` — tomorrow `matters.map(m => <MatterCard ... />)` renders one card per matter in React. Same method, UI instead of strings.

### 6 · The DOM & events — how a page reacts (browser-only)

In a browser, JS reaches into the page via the **DOM** (`document.querySelector(...)`) and responds to **events** like clicks (`onclick` / `addEventListener`). This is the one part that needs a real browser — Jupyter sandboxes `<script>`, so the button below *renders* but won't *run* here.

In [ ]:
demo = """
<div style="font-family:sans-serif;padding:12px;border:1px solid #e5e7eb;border-radius:8px;max-width:360px;">
  <strong>Active matters:</strong> <span id="count">3</span>
  <button onclick="document.getElementById('count').textContent='2'">Close one</button>
</div>
<script>
  // Runs in a browser (open the saved file); ignored in notebook output.
  console.log("matters UI ready");
</script>
"""
with open("events_demo.html", "w") as f:
    f.write("<!DOCTYPE html><html><body>" + demo + "</body></html>")
print("📄 Saved events_demo.html — open it and click the button to see JS update the page.")
show(demo)

**What just happened:** the `onclick` wires a click to a bit of JS that changes the count. It's inert in the notebook (by design) but live when you open `events_demo.html`. **Events are how every button, filter, and toggle in an artifact works** — and React gives us a tidier way to write them next.

> **`Go Deeper 🔧` — `===`, and JSON.** Use **`===`** (strict equals), not `==` — `==` does surprising type coercion (`0 == ""` is `true`!). And **`JSON.parse(text)`** / **`JSON.stringify(obj)`** convert between a JSON string and JS objects — the exact bridge you'll formalize in Week 5 when data arrives over HTTP.

In [ ]:
run_js("""
console.log(0 == "");    // true  (loose — avoid)
console.log(0 === "");   // false (strict — use this)

const obj = { id: "M-1001", billed: 18500 };
const text = JSON.stringify(obj);         // object -> JSON string
console.log(text);
console.log(JSON.parse(text).id);         // JSON string -> object
""")

> **`Common pitfalls ⚠️`**
>
> - **`const` can't be reassigned** — use `let` when a value changes. (You *can* still edit an object/array declared with `const`.)
> - **`===` not `==`** — strict equality avoids type-coercion surprises.
> - **Semicolons & braces**, not indentation — the block is `{ … }`.
> - **JS runs in the browser (or Node), not in notebook HTML output** — save the file and open it to see events fire.

### ✍️ Your turn

Edit the JS and run it with `run_js`.

In [ ]:
run_js("""
const matters = [
  { id: "M-1002", client: "Brightline LLC", area: "Litigation", billed: 42750.5, active: true  },
  { id: "M-1001", client: "Acme Corp",      area: "Contracts",  billed: 18500.0, active: true  },
  { id: "M-1004", client: "Dovetail Inc",   area: "Employment", billed: 9800.0,  active: true  },
  { id: "M-1003", client: "Cedar Holdings", area: "M&A",        billed: 131200.0, active: false },
];

// TODO 1: log the number of CLOSED matters (filter m.active === false)
// TODO 2: map matters to an array of just their `client` names, and log it
// TODO 3: reduce to the total billed for ACTIVE matters only
//         (hint: filter first, then reduce)

""")

<details><summary>✅ Show solution</summary>

```python
run_js("""
const matters = [
  { id: "M-1002", client: "Brightline LLC", billed: 42750.5, active: true  },
  { id: "M-1001", client: "Acme Corp",      billed: 18500.0, active: true  },
  { id: "M-1004", client: "Dovetail Inc",   billed: 9800.0,  active: true  },
  { id: "M-1003", client: "Cedar Holdings", billed: 131200.0, active: false },
];

// 1
console.log("closed:", matters.filter(m => m.active === false).length);

// 2
console.log(matters.map(m => m.client));

// 3
const activeTotal = matters.filter(m => m.active).reduce((s, m) => s + m.billed, 0);
console.log("active billed:", activeTotal.toFixed(2));
""")
```
</details>

### 🚀 Build the artifact — a matters summary module

Write a small JavaScript module and run it with Node — the kind of logic that lives in an artifact's `<script>`. It computes exactly the numbers a dashboard header shows.

In [ ]:
run_js("""
const matters = [
  { id: "M-1002", client: "Brightline LLC", area: "Litigation", billed: 42750.5, active: true  },
  { id: "M-1001", client: "Acme Corp",      area: "Contracts",  billed: 18500.0, active: true  },
  { id: "M-1004", client: "Dovetail Inc",   area: "Employment", billed: 9800.0,  active: true  },
  { id: "M-1003", client: "Cedar Holdings", area: "M&A",        billed: 131200.0, active: false },
];

const summarize = (rows) => {
  const active = rows.filter(m => m.active);
  const total  = rows.reduce((s, m) => s + m.billed, 0);
  return {
    matters: rows.length,
    active: active.length,
    totalBilled: total,
    topClient: [...rows].sort((a, b) => b.billed - a.billed)[0].client,
  };
};

const s = summarize(matters);
console.log(`📊 ${s.active}/${s.matters} active · $${s.totalBilled.toFixed(2)} billed · top: ${s.topClient}`);
""")

> **🔗 Your world.** That `summarize()` is the logic behind a dashboard's header stats. Right now the data is hardcoded; in **Week 5** it arrives as JSON over HTTP from our Snowflake matters. And that `filter`/`map`/`reduce` fluency is the real prize — **tomorrow `matters.map(...)` builds React components instead of strings**, turning this data into a live UI.

### 📝 Recap — what you shipped

- JavaScript = the **behavior** layer; `const`/`let`, `` `${…}` `` template literals, arrow `=>` functions.
- Web data is an **array of objects** (`[{...}]`) — a list of records.
- **`filter` / `map` / `reduce`** transform arrays — and **`map` is how React renders lists**.
- The **DOM + events** let a page react (browser-only; Jupyter sandboxes `<script>`).
- **Artifact:** a `summarize()` module run with Node, plus `events_demo.html`.

### 🧠 Check your understanding

1. When do you use `const` vs `let`?
2. What does `matters.map(m => m.client)` produce?
3. Which array method sums a field, and what's the `0` for in `reduce((s, m) => s + m.billed, 0)`?
4. Why doesn't the `onclick` button do anything in the notebook, and how do you see it work?

<details><summary>✅ Answers</summary>

1. `const` for values you won't reassign (most things); `let` when the variable will change.
2. A **new array** of each matter's `client` name (same length, transformed items).
3. **`reduce`**; the `0` is the **starting value** of the accumulator `s` (the running total begins at 0).
4. Jupyter **sandboxes `<script>`** for safety; **open the saved `.html` in a browser** to run it.
</details>

### ➡️ Next up — Week 4, Day 2: React & JSX

You can transform data with `map`; React uses that exact move to turn matters into **components**. Tomorrow we read **JSX** (HTML-inside-JavaScript), meet **props** and **`useState`**, and understand the Claude artifacts written in React — the ones you'll soon port into a real project.

*Same `run_js`/`show` setup, no install.*

### 📖 Reference & glossary

| Term | Plain meaning | Python twin |
|---|---|---|
| **`const` / `let`** | declare a constant / a variable | `x = …` |
| **`console.log`** | print | `print` |
| **Template literal** | `` `text ${var}` `` | f-string |
| **Arrow function** | `(a) => a + 1` | `lambda a: a+1` |
| **Object / array** | `{k: v}` / `[…]` | `dict` / `list` |
| **`filter`** | keep matching items | `[x for x in xs if …]` |
| **`map`** | transform each item | `[f(x) for x in xs]` |
| **`reduce`** | collapse to one value | `sum(...)` / `functools.reduce` |
| **DOM / event** | the page as objects / a user action | — |

**Docs:** MDN — JavaScript first steps: https://developer.mozilla.org/en-US/docs/Learn/JavaScript/First_steps · Array methods: https://developer.mozilla.org/en-US/docs/Web/JavaScript/Reference/Global_Objects/Array

> *Not legal advice — these lessons teach technology. A lawyer reviews any AI output that will be relied upon.*